<a href="https://colab.research.google.com/github/Lakshmi-Muraleedharan/100NextGen_v1/blob/main/Assignment%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install langchain langchain-core langchain-community langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 3.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 4.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.3 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   

In [2]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_988/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
#Part1
loader=PyPDFLoader("/content/troubleshooting_runbook.pdf")

In [6]:
!pip install pypdf

In [7]:
documents=loader.load()
report=""
for page in documents:
  report+=page.page_content+'\n'
print("Total no of pages",len(documents))
print(report[:500])

Total no of pages 7
Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 1
Contoso Platform Troubleshooting Runbook
Document owner: Platform Reliability Team  |  Version: 3.2  |  Last reviewed: 2026-10-15
1. Purpose and scope
This runbook gives on-call engineers a consistent way to diagnose and resolve production incidents
across the eight core Contoso services: payment-gateway, auth-service, order-api, inventory-service,
notification-service, search-service, reporting-e


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
)
chunk=text_splitter.split_documents(documents)
print(len(chunk))

26


In [9]:
pip install langchain_community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 36.2 MB/s  0:00:00


In [10]:
#Part2
from  langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [11]:
import os
import getpass

In [14]:
os.environ["GEMINI_API_KEY"]=getpass.getpass("Enter API Key")

Enter API Key··········


In [15]:
embedding=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [16]:
vector_store=FAISS.from_documents(
    documents=chunk,
    embedding=embedding
)

In [17]:
retriver=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":2}
)

In [18]:
#Part 4
from langchain_core.prompts import ChatPromptTemplate


In [19]:
prompt=ChatPromptTemplate.from_template(
    """
    You are a Q&A assistant who answers questions on troubleshooting runbook.
    Please provide response based on following rules:
    - Use only the data provided below to generate the response
    - Donot assume or use general information to generate response
    - If the answer cannot be found in the provided playbook, state that the information is not available in
the document rather than generating an answer from general knowledge
    - Keep the answer consise and complete
    - If the data contains page reference, include it in the response
    context:{context}
    question:{question}
    Provode the following:
    1. Answer
    2. Supporting evidence
    """

)

In [20]:
from langchain_core.output_parsers import StrOutputParser


In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(model='gemini-3.6-flash')
chain=prompt|llm|StrOutputParser()

In [22]:
question="What policy does API gateway use to enforce ratelimiting"

In [23]:
retrived_docs=retriver.invoke(question)
context="\n\n".join(
    [doc.page_content for doc in retrived_docs])
response=chain.invoke({"context":context,"question":question})
print(response)

**1. Answer**
The API gateway uses a token-bucket policy per API key to enforce rate limiting.

**2. Supporting evidence**
"Throttling is enforced at the API gateway using a token-bucket policy per API key."


In [24]:
question="What response header is sent wuth 429 error code"

In [ ]:
retrived_docs=retriver.invoke(question)
context="\n\n".join(
    [doc.page_content for doc in retrived_docs])
response=chain.invoke({"context":context,"question":question})
print(response)